# Silver Layer


fhir

In [0]:
bronze_df = spark.read.table("bronze.fhir_raw")

In [0]:
bronze_df.printSchema()

one row contain array of struct so we will explode the data so that one row contain one struct only

In [0]:
from pyspark.sql.functions import explode

In [0]:
resource_df = bronze_df.select(explode("entry").alias("entry"))

## Patient Record

In [0]:
patient_df = resource_df.filter(resource_df["entry.resource.resourceType"] == 'Patient')

In [0]:
# patient_df.printSchema()

In [0]:
from pyspark.sql.functions import col

patient_fhir_df = patient_df.select(
    col("entry.resource.id").alias("Patient_Id"),
    col("entry.resource.birthDate").alias("Birth_date"),
    col("entry.resource.gender").alias("Gender"),
    col("entry.resource.name").alias("Name")
)

display(patient_fhir_df)


In [0]:
from pyspark.sql.functions import get_json_object

In [0]:
patient_fhir_df = patient_fhir_df.withColumn(
    "given_name",
    get_json_object("name", "$[0].given[0]")
).withColumn(
    "family_name",
    get_json_object("name", "$[0].family")
)

In [0]:

patient_fhir_df.select(
    "given_name",
    "family_name"
).show(5, truncate=False)



In [0]:
patient_fhir_df = patient_fhir_df.drop(
    "name"
)

In [0]:
from pyspark.sql.functions import concat_ws

patient_fhir_df = patient_fhir_df.withColumn(
    "full_name",
    concat_ws(" ", "given_name", "family_name")
)

In [0]:
display(patient_fhir_df)

## Encourter Records

In [0]:
encounter_df = resource_df.filter(resource_df["entry.resource.resourceType"] == 'Encounter')

encounter_fhir_df= encounter_df.select(
    col("entry.resource.id").alias("Encounter_Id"),
    col("entry.resource.status").alias("Status"),
    col("entry.resource.class.code").alias("Class"),
    col("entry.resource.subject.reference").alias("Patient_Reference"),
    col("entry.resource.subject.display").alias("Patient_Name"),
    col("entry.resource.period.start").alias("Start_Date"),
    col("entry.resource.period.end").alias("End_Date")
)

In [0]:
display(encounter_fhir_df)

## Condition Records

In [0]:
from pyspark.sql.functions import get_json_object

In [0]:
condition_df = resource_df.filter(resource_df["entry.resource.resourceType"]=="Condition")

condition_fhir_df = condition_df.select(
    col("entry.resource.id").alias("Condition_Id"),
    col("entry.resource.clinicalStatus.coding.code")[0].alias("Clinical_Status"),
    col("entry.resource.verificationStatus.coding.code")[0].alias("Verification_Status"),
    get_json_object(
        col("entry.resource.code"),
        "$.coding[0].code"
    ).alias("Condition_Code"),

   
    get_json_object(
        col("entry.resource.code"),
        "$.coding[0].display"
    ).alias("Condition_Name"),

    get_json_object(
        col("entry.resource.category")[0],
        "$.coding[0].display"
    ).alias("Category"),
   
    col("entry.resource.subject.reference").alias("Patient_Reference"),
    col("entry.resource.encounter.reference").alias("Encounter_Reference"),
    col("entry.resource.recordedDate").alias("Recorded_Date"),
    col("entry.resource.onsetDateTime").alias("Onset_Date")
)

display(condition_fhir_df)



## Observation Resource

In [0]:
observation_df = resource_df.filter(resource_df["entry.resource.resourceType"]=="Observation")



In [0]:
observation_fhir_df = observation_df.select(
    col("entry.resource.id").alias("Observation_Id"),
    col("entry.resource.status").alias("Status"),

    get_json_object(
        col("entry.resource.category")[0],
        "$.coding[0].display"
    ).alias("Observation_Name"),

    col("entry.resource.subject.reference").alias("Patient_Reference"),
    col("entry.resource.encounter.reference").alias("Encounter_Reference"),
    to_date(col("entry.resource.effectiveDateTime")).alias("Observation_Date"),
    to_date(col("entry.resource.issued")).alias("Issued_Date"),
    col("entry.resource.valueQuantity.value").cast("string").alias("Value"),
    col("entry.resource.valueQuantity.unit").alias("Unit")
)

display(observation_fhir_df)

## Medication Request Resource

In [0]:
medicationRequest_df = resource_df.filter(resource_df["entry.resource.resourceType"]=="MedicationRequest")

In [0]:
medicationRequest_fhir_df = medicationRequest_df.select(
    col("entry.resource.id").alias("MedicationRequest_Id"),
    col("entry.resource.status").alias("Status"),
    col("entry.resource.intent").alias("Intent"),
    col("entry.resource.subject.reference").alias("Patient_Reference"),
    col("entry.resource.medicationCodeableConcept.coding.code")[0].alias("Medication_Code"),
    col("entry.resource.medicationCodeableConcept.coding.display")[0].alias("Medication_Name"),
    col("entry.resource.encounter.reference").alias("Encounter_Reference"),
    col("entry.resource.authoredOn").alias("Authored_On"),
    col("entry.resource.requester.reference").alias("Requester_Reference"),
    col("entry.resource.requester.display").alias("Requester_Name"),
    
)


display(medicationRequest_fhir_df)

##Deduplication logic using Resource IDs as surrogate key

In [0]:
patient_fhir_df = patient_fhir_df.dropDuplicates(["Patient_ID"])
encounter_fhir_df = encounter_fhir_df.dropDuplicates(["Encounter_ID"])
condition_fhir_df = condition_fhir_df.dropDuplicates(["Condition_ID"])
observation_fhir_df = observation_fhir_df.dropDuplicates(["Observation_ID"])
medicationRequest_fhir_df = medicationRequest_fhir_df.dropDuplicates(["MedicationRequest_ID"])

## Delta Live Tables

In [0]:
from pyspark import pipelines as dp

@dp.table(name="patient_silver_dlt")
@dp.expect("valid_patient_id", "Patient_Id IS NOT NULL")
@dp.expect("valid_birth_date", "Birth_date IS NOT NULL")
@dp.expect("valid_gender", "Gender IN ('male','female')")
def patient_silver():
    return patient_fhir_df


@dp.table(name="encounter_silver_dlt")
@dp.expect("valid_encounter_id", "Encounter_Id IS NOT NULL")
@dp.expect("valid_patient_reference", "Patient_Reference IS NOT NULL")
@dp.expect("valid_encounter_status", "Status IS NOT NULL")
def encounter_silver():
    return encounter_fhir_df


@dp.table(name="condition_silver_dlt")
@dp.expect("valid_condition_id", "Condition_Id IS NOT NULL")
@dp.expect("valid_patient_reference", "Patient_Reference IS NOT NULL")
@dp.expect("valid_condition_name", "Condition_Name IS NOT NULL")
def condition_silver():
    return condition_fhir_df


@dp.table(name="observation_silver_dlt")
@dp.expect("valid_observation_id", "Observation_Id IS NOT NULL")
@dp.expect("valid_patient_reference", "Patient_Reference IS NOT NULL")
@dp.expect("valid_status", "Status IS NOT NULL")

def observation_silver():
    return observation_fhir_df


@dp.table(name="medication_request_silver_dlt")
@dp.expect("valid_medication_request_id", "MedicationRequest_Id IS NOT NULL")
@dp.expect("valid_medication_name", "Medication_Name IS NOT NULL")
@dp.expect("valid_patient_reference", "Patient_Reference IS NOT NULL")

def medication_request_silver():
    return medicationRequest_fhir_df

DLT data quality expectations identified 354 MedicationRequest records with missing Medication_Name values (1.5% of records). These records were logged as quality violations while still being retained in the Silver layer.

hl7

In [0]:
hl7_df = spark.read.table("bronze.hl7_adt_raw")

In [0]:
hl7_df.printSchema()

In [0]:
display(hl7_df)

In [0]:
from pyspark.sql.functions import split, explode

hl7_segments = hl7_df.withColumn(
    "segments",
    split("raw_message", "\n")
)

segment_df = hl7_segments.select(
    "path",
    explode("segments").alias("segment")
)

In [0]:
from pyspark.sql.functions import explode

segment_df = hl7_segments.select(
    "path",
    explode("segments").alias("segment")
)

In [0]:
from pyspark.sql.functions import split

segment_df = segment_df.withColumn(
    "segment_type",
    split("segment", "\\|").getItem(0)
)

In [0]:
patient_segment = segment_df.filter("segment_type='PID'")

In [0]:
from pyspark.sql.functions import split

patient_hl7_df = patient_segment.withColumn(
    "fields",
    split("segment", "\\|")
)

In [0]:
patient_hl7_df.select("fields").show(1, truncate=False)

In [0]:
from pyspark.sql.functions import col, split, concat_ws, get

patient_hl7_df = (
    patient_segment
    .withColumn("fields", split(col("segment"), "\\|"))
    .select(
        get(col("fields"), 3).alias("patient_id"),
        get(col("fields"), 5).alias("patient_name"),
        get(col("fields"), 7).alias("birth_date"),
        get(col("fields"), 8).alias("gender"),
        get(col("fields"), 11).alias("address"),
        get(col("fields"), 13).alias("phone")
    )
    .withColumn("name_parts", split(col("patient_name"), "\\^"))
    .withColumn("family_name", get(col("name_parts"), 0))
    .withColumn("given_name", get(col("name_parts"), 1))
    .withColumn("middle_name", get(col("name_parts"), 2))
    .withColumn(
        "full_name",
        concat_ws(" ", col("given_name"), col("middle_name"), col("family_name"))
    )
)

In [0]:
display(patient_hl7_df)

In [0]:
pv1_segment = segment_df.filter(col("segment_type") == "PV1")

In [0]:
encounter_hl7_df = (
    pv1_segment
    .withColumn("fields", split(col("segment"), "\\|"))
)

display(encounter_hl7_df)

In [0]:
encounter_hl7_df = encounter_hl7_df.select(
    get(col("fields"),19).alias("encounter_id"),
    get(col("fields"),2).alias("patient_class"),
    get(col("fields"),3).alias("location"),
    get(col("fields"),7).alias("provider"),
    get(col("fields"),4).alias("admission_type"),
    get(col("fields"),10).alias("hospital_service")
)

In [0]:
display(encounter_hl7_df)

flat files

In [0]:
patient_flat_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("abfss://raw@sthealthcarekl.dfs.core.windows.net/flat files/patients.csv")
)

encounter_flat_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("abfss://raw@sthealthcarekl.dfs.core.windows.net/flat files/encounters.csv")
)

condition_flat_df = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("abfss://raw@sthealthcarekl.dfs.core.windows.net/flat files/conditions.csv")
)

observation_flat_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("abfss://raw@sthealthcarekl.dfs.core.windows.net/flat files/observations.csv")
)

medication_flat_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("abfss://raw@sthealthcarekl.dfs.core.windows.net/flat files/medications.csv")
)

In [0]:
patient_flat_df.printSchema()

encounter_flat_df.printSchema()

condition_flat_df.printSchema()

observation_flat_df.printSchema()

medication_flat_df.printSchema()

In [0]:
display(patient_flat_df)

display(encounter_flat_df)

display(condition_flat_df)

display(observation_flat_df)

display(medication_flat_df)

In [0]:
from pyspark.sql.functions import concat_ws

patient_flat_df = patient_flat_df.select(
    patient_flat_df.Id.alias("patient_id"),
    concat_ws(" ",
              patient_flat_df.FIRST,
              patient_flat_df.MIDDLE,
              patient_flat_df.LAST).alias("full_name"),
    patient_flat_df.BIRTHDATE.alias("birth_date"),
    patient_flat_df.GENDER.alias("gender"),
    patient_flat_df.ADDRESS.alias("address"),
    patient_flat_df.CITY.alias("city"),
    patient_flat_df.STATE.alias("state"),
    patient_flat_df.ZIP.alias("zip")
)

display(patient_flat_df)

In [0]:
encounter_flat_df = encounter_flat_df.select(
    encounter_flat_df.Id.alias("encounter_id"),
    encounter_flat_df.PATIENT.alias("patient_id"),
    encounter_flat_df.START.alias("start_time"),
    encounter_flat_df.STOP.alias("end_time"),
    encounter_flat_df.ENCOUNTERCLASS.alias("encounter_class"),
    encounter_flat_df.DESCRIPTION.alias("description"),
    encounter_flat_df.ORGANIZATION.alias("organization"),
    encounter_flat_df.PROVIDER.alias("provider")
)

display(encounter_flat_df)

In [0]:
from pyspark.sql.functions import col, lit

condition_flat_df = condition_flat_df.select(
    lit(None).cast("string").alias("Condition_Id"),
    lit(None).cast("string").alias("Clinical_Status"),
    lit(None).cast("string").alias("Verification_Status"),
    col("CODE").alias("Condition_Code"),
    col("DESCRIPTION").alias("Condition_Name"),
    lit(None).cast("string").alias("Category"),
    col("PATIENT").alias("Patient_Reference"),
    col("ENCOUNTER").alias("Encounter_Reference"),
    col("START").alias("Recorded_Date"),
    col("START").alias("Onset_Date")
)

In [0]:
observation_flat_df = observation_flat_df.select(
    observation_flat_df.PATIENT.alias("patient_id"),
    observation_flat_df.ENCOUNTER.alias("encounter_id"),
    observation_flat_df.CODE.alias("observation_code"),
    observation_flat_df.DESCRIPTION.alias("observation_name"),
    observation_flat_df.VALUE.alias("value"),
    observation_flat_df.UNITS.alias("unit"),
    observation_flat_df.DATE.alias("observation_date")
)

display(observation_flat_df)

In [0]:
medication_flat_df = medication_flat_df.select(
    medication_flat_df.PATIENT.alias("patient_id"),
    medication_flat_df.ENCOUNTER.alias("encounter_id"),
    medication_flat_df.CODE.alias("medication_code"),
    medication_flat_df.DESCRIPTION.alias("medication_name"),
    medication_flat_df.START.alias("start_date"),
    medication_flat_df.STOP.alias("end_date"),
    medication_flat_df.TOTALCOST.alias("total_cost")
)

display(medication_flat_df)

union all

In [0]:
patient_fhir_df = patient_fhir_df.select(
    "patient_id",
    "full_name",
    "birth_date",
    "gender",
    "address",
    "phone"
)

patient_hl7_df = patient_hl7_df.select(
    "patient_id",
    "full_name",
    "birth_date",
    "gender",
    "address",
    "phone"
)

from pyspark.sql.functions import concat_ws, col

patient_flat_df = patient_flat_df.select(
    col("Id").alias("patient_id"),
    concat_ws(" ", col("FIRST"), col("MIDDLE"), col("LAST")).alias("full_name"),
    col("BIRTHDATE").alias("birth_date"),
    col("GENDER").alias("gender"),
    col("ADDRESS").alias("address"),
    col("DRIVERS").alias("phone")   
)

In [0]:
patient_silver_df = (
    patient_fhir_df
    .unionByName(patient_hl7_df)
    .unionByName(patient_flat_df)
)

In [0]:
from pyspark.sql.functions import col, lit

encounter_flat_df = (
    encounter_flat_df
    .select(
        col("Id").alias("encounter_id"),
        col("PATIENT").alias("patient_reference"),
        col("START").alias("start_date"),
        col("STOP").alias("end_date"),
        col("ENCOUNTERCLASS").alias("patient_class"),
        col("PROVIDER").alias("provider"),
        col("ORGANIZATION").alias("organization"),
        col("DESCRIPTION").alias("reason"),
        col("REASONDESCRIPTION").alias("reason_description")
    )
)

In [0]:
from pyspark.sql.functions import lit

encounter_hl7_df = (
    encounter_hl7_df
    .withColumn("patient_reference", lit(None))
    .withColumn("start_date", lit(None))
    .withColumn("end_date", lit(None))
    .withColumn("organization", lit(None))
    .withColumn("reason", lit(None))
    .withColumn("reason_description", lit(None))
)

In [0]:
encounter_hl7_df = encounter_hl7_df.select(
    "encounter_id",
    "patient_reference",
    "start_date",
    "end_date",
    "patient_class",
    "provider",
    "organization",
    "reason",
    "reason_description"
)

In [0]:
encounter_fhir_df = encounter_fhir_df.select(
    "encounter_id",
    "patient_reference",
    "start_date",
    "end_date",
    "patient_class",
    "provider",
    "organization",
    "reason",
    "reason_description"
)

In [0]:
encounter_silver_df = (
    encounter_fhir_df
        .unionByName(encounter_hl7_df)
        .unionByName(encounter_flat_df)
)

In [0]:
from pyspark.sql.functions import col, lit

observation_flat_df = (
    observation_flat_df.select(
        col("patient_id").alias("Patient_Reference"),
        col("encounter_id").alias("Encounter_Reference"),
        col("observation_name").alias("Observation_Name"),
        col("observation_date").alias("Observation_Date"),
        col("observation_date").alias("Issued_Date"),
        col("value").alias("Value"),
        col("unit").alias("Unit"),
        lit(None).cast("string").alias("Observation_Id"),
        lit("final").alias("Status")
    )
)

In [0]:
display(observation_flat_df)

In [0]:
observation_silver_df = (
    observation_fhir_df
    .unionByName(observation_flat_df)
)

In [0]:
display(observation_silver_df)

In [0]:
condition_flat_df = condition_flat_df.select(
    lit(None).cast("string").alias("Condition_Id"),
    col("PATIENT").alias("Patient_Reference"),
    col("ENCOUNTER").alias("Encounter_Reference"),
    col("CODE").alias("Condition_Code"),
    col("DESCRIPTION").alias("Condition_Name"),
    lit(None).cast("string").alias("Category"),
    col("START").alias("Recorded_Date"),
    col("START").alias("Onset_Date"),
    lit(None).cast("string").alias("Clinical_Status"),
    lit(None).cast("string").alias("Verification_Status")
)

In [0]:
condition_flat_df = condition_flat_df.select(
    "Condition_Id",
    "Clinical_Status",
    "Verification_Status",
    "Condition_Code",
    "Condition_Name",
    "Category",
    "Patient_Reference",
    "Encounter_Reference",
    "Recorded_Date",
    "Onset_Date"
)

In [0]:
condition_silver_df = (
    condition_fhir_df
    .unionByName(condition_flat_df)
)

In [0]:
display(condition_silver_df)